# **Random forest**

In [1]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv(
    "spotify_songs.csv",
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    encoding='utf-8',
    engine='python'
)

In [74]:
df.head()

,track_id,track_name,track_artist,lyrics,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language
0,0017A6SJgTbfQVU2EtsPNo,Pangarap,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,41,1srJQ0njEQgd8w4XSqI4JQ,Trip,2001-01-01,Pinoy Classic Rock,37i9dQZF1DWYDQ8wBxd7xt,...,-10.068,1.0,0.0236,0.279,0.01170,0.0887,0.566,97.091,235440.0,tl
1,00emjlCv9azBN0fzuuyLqy,Dumb Litty,KARD,Get up out of my business You don't keep me fr...,65,7h5X3xhh3peIK9Y0qI5hbK,KARD 2nd Digital Single ‘Dumb Litty’,2019-09-22,K-Party Dance Mix,37i9dQZF1DX4RDXswvP6Mj,...,-1.993,1.0,0.0409,0.037,0.00000,0.1380,0.240,130.018,193160.0,en
2,00hdjyXt6MohKnCyDmhxOL,Una Vaina Loca,Fuego,Fuego Uoh uoh uoh La musiica del futuroo Yeah!...,1,01nV3KuocS1NJHTsJbPkTO,Una Vaina Loca,2011-11-02,MIX LATIN POP°,6IS6XTdbS9qJZgfjNKgpB8,...,-5.589,1.0,0.0361,0.114,0.00008,0.0620,0.642,117.009,188213.0,es
3,00PLtXXER1XcTRZvs3LioS,Get The Funk Out Ma Face,The Brothers Johnson,Get the funk out ma face Get the funk out ma f...,49,5IzEY3pod97kHrdt5Qt1RB,Look Out For #1,1976-01-01,The 1950s/1960s/1970s/1980s/1990s/2000s/2010s ...,1S7BckuYIkEazeNKOSM0uA,...,-13.145,1.0,0.0821,0.292,0.03880,0.0674,0.791,106.035,147000.0,en
4,00vHOSgvVfyLzhlTZIJo1m,Full Moon,DREAMCATCHER,드림캐쳐 Full Moon 가사Hangul 시간을 따라 걸으면 조금씩 같이 걸으면 ...,49,6N7UA00H0IAexiQvOFGv5d,Full Moon,2018-01-12,K-Crazy Michioso Tunes,37i9dQZF1DWUXxc8Mc6MmJ,...,-2.112,1.0,0.1130,0.186,0.00000,0.3810,0.248,94.964,189711.0,et


In [ ]:
drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

df.head()

,track_artist,lyrics,track_popularity,track_album_id,track_album_release_date,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language
0,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,41,1srJQ0njEQgd8w4XSqI4JQ,2001-01-01,rock,classic rock,0.682,0.401,2.0,-10.068,1.0,0.0236,0.279,0.01170,0.0887,0.566,97.091,235440.0,tl
1,KARD,Get up out of my business You don't keep me fr...,65,7h5X3xhh3peIK9Y0qI5hbK,2019-09-22,pop,dance pop,0.760,0.887,9.0,-1.993,1.0,0.0409,0.037,0.00000,0.1380,0.240,130.018,193160.0,en
2,Fuego,Fuego Uoh uoh uoh La musiica del futuroo Yeah!...,1,01nV3KuocS1NJHTsJbPkTO,2011-11-02,latin,latin pop,0.794,0.882,1.0,-5.589,1.0,0.0361,0.114,0.00008,0.0620,0.642,117.009,188213.0,es
3,The Brothers Johnson,Get the funk out ma face Get the funk out ma f...,49,5IzEY3pod97kHrdt5Qt1RB,1976-01-01,r&b,urban contemporary,0.880,0.570,2.0,-13.145,1.0,0.0821,0.292,0.03880,0.0674,0.791,106.035,147000.0,en
4,DREAMCATCHER,드림캐쳐 Full Moon 가사Hangul 시간을 따라 걸으면 조금씩 같이 걸으면 ...,49,6N7UA00H0IAexiQvOFGv5d,2018-01-12,edm,pop edm,0.589,0.945,7.0,-2.112,1.0,0.1130,0.186,0.00000,0.3810,0.248,94.964,189711.0,et


In [ ]:

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df.drop(columns=['playlist_genre', 'genre_encoded'])
y = df['genre_encoded']

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)

print("✅ Accuracy:", accuracy)
print("✅ Precision:", precision)
print("✅ F1 Score:", f1)
print("✅ Confusion Matrix:\n", conf_matrix)
print("\n✅ Classification Report:\n", report)

✅ Accuracy: 0.8160377358490566
✅ Precision: 0.8695550896808759
✅ F1 Score: 0.786170417304152
✅ Confusion Matrix:
 [[55  0  0  0  0  1]
 [ 1 18  0  0  0  8]
 [ 1  0 27  0  0  0]
 [ 1  0  0 19  0 10]
 [ 8  0  0  0  1  9]
 [ 0  0  0  0  0 53]]

✅ Classification Report:
               precision    recall  f1-score   support

         edm       0.83      0.98      0.90        56
       latin       1.00      0.67      0.80        27
         pop       1.00      0.96      0.98        28
         r&b       1.00      0.63      0.78        30
         rap       1.00      0.06      0.11        18
        rock       0.65      1.00      0.79        53

    accuracy                           0.82       212
   macro avg       0.91      0.72      0.73       212
weighted avg       0.87      0.82      0.79       212



# **KNN**

In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv(
    "spotify_songs.csv",
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    encoding='utf-8',
    engine='python'
)

drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df.drop(columns=['playlist_genre', 'genre_encoded'])
y = df['genre_encoded']

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)

print(" Accuracy:", accuracy)
print(" Precision:", precision)
print(" F1 Score:", f1)
print(" Confusion Matrix:\n", conf_matrix)
print("\n Classification Report:\n", report)


✅ Accuracy: 0.3247863247863248
✅ Precision: 0.30837756616092343
✅ F1 Score: 0.3038273776052199
✅ Confusion Matrix:
 [[37  4  9  5  1 10]
 [10  3  5  1  1  7]
 [16  0 13  3  2  8]
 [11  2  2  3  1  8]
 [ 6  0  5  3  2  0]
 [25  3  7  3  0 18]]

✅ Classification Report:
               precision    recall  f1-score   support

         edm       0.35      0.56      0.43        66
       latin       0.25      0.11      0.15        27
         pop       0.32      0.31      0.31        42
         r&b       0.17      0.11      0.13        27
         rap       0.29      0.12      0.17        16
        rock       0.35      0.32      0.34        56

    accuracy                           0.32       234
   macro avg       0.29      0.26      0.26       234
weighted avg       0.31      0.32      0.30       234



# **SVM**

In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv(
    "spotify_songs.csv",
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    encoding='utf-8',
    engine='python'
)

drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df.drop(columns=['playlist_genre', 'genre_encoded'])
y = df['genre_encoded']

X = pd.get_dummies(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)

print(" Accuracy:", accuracy)
print(" Precision:", precision)
print(" F1 Score:", f1)
print(" Confusion Matrix:\n", conf_matrix)
print("\n Classification Report:\n", report)


✅ Accuracy: 0.41452991452991456
✅ Precision: 0.4743396372355649
✅ F1 Score: 0.31382322511439636
✅ Confusion Matrix:
 [[40  0  0  0  0 26]
 [ 2  0  0  0  0 25]
 [ 2  0  1  0  0 39]
 [ 5  0  0  0  0 22]
 [ 2  0  0  0  0 14]
 [ 0  0  0  0  0 56]]

✅ Classification Report:
               precision    recall  f1-score   support

         edm       0.78      0.61      0.68        66
       latin       0.00      0.00      0.00        27
         pop       1.00      0.02      0.05        42
         r&b       0.00      0.00      0.00        27
         rap       0.00      0.00      0.00        16
        rock       0.31      1.00      0.47        56

    accuracy                           0.41       234
   macro avg       0.35      0.27      0.20       234
weighted avg       0.47      0.41      0.31       234



# Naive Bayes 

In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv(
    "spotify_songs.csv",
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    encoding='utf-8',
    engine='python'
)

drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df.drop(columns=['playlist_genre', 'genre_encoded'])
y = df['genre_encoded']

X = pd.get_dummies(X)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)

print(" Accuracy:", accuracy)
print(" Precision:", precision)
print(" F1 Score:", f1)
print(" Confusion Matrix:\n", conf_matrix)
print("\n Classification Report:\n", report)


✅ Accuracy: 0.8888888888888888
✅ Precision: 0.8392935230144533
✅ F1 Score: 0.857253098991169
✅ Confusion Matrix:
 [[66  0  0  0  0  0]
 [ 2 20  0  0  0  5]
 [ 0  0 42  0  0  0]
 [ 0  0  0 24  0  3]
 [ 9  0  1  0  0  6]
 [ 0  0  0  0  0 56]]

✅ Classification Report:
               precision    recall  f1-score   support

         edm       0.86      1.00      0.92        66
       latin       1.00      0.74      0.85        27
         pop       0.98      1.00      0.99        42
         r&b       1.00      0.89      0.94        27
         rap       0.00      0.00      0.00        16
        rock       0.80      1.00      0.89        56

    accuracy                           0.89       234
   macro avg       0.77      0.77      0.77       234
weighted avg       0.84      0.89      0.86       234



# **linear regression**

In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report, r2_score, mean_squared_error

df = pd.read_csv(
    "spotify_songs.csv",
    on_bad_lines='skip',
    quoting=csv.QUOTE_NONE,
    encoding='utf-8',
    engine='python'
)

drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df.drop(columns=['playlist_genre', 'genre_encoded'])
y = df['genre_encoded']

X = pd.get_dummies(X)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

nb = MultinomialNB()
nb.fit(X_train, y_train)


y_pred = nb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_lr_pred = lr.predict(X_test)

r2 = r2_score(y_test, y_lr_pred)
mse = mean_squared_error(y_test, y_lr_pred)

print(" Naive Bayes Evaluation:")
print(" Accuracy:", accuracy)
print(" Precision:", precision)
print(" F1 Score:", f1)
print(" Confusion Matrix:\n", conf_matrix)
print("\n Classification Report:\n", report)

# Linear Regression Evaluation
print("\n Linear Regression Evaluation:")
print(" R-squared:", r2)
print(" Mean Squared Error:", mse)


✅ Naive Bayes Evaluation:
✅ Accuracy: 0.8888888888888888
✅ Precision: 0.8392935230144533
✅ F1 Score: 0.857253098991169
✅ Confusion Matrix:
 [[66  0  0  0  0  0]
 [ 2 20  0  0  0  5]
 [ 0  0 42  0  0  0]
 [ 0  0  0 24  0  3]
 [ 9  0  1  0  0  6]
 [ 0  0  0  0  0 56]]

✅ Classification Report:
               precision    recall  f1-score   support

         edm       0.86      1.00      0.92        66
       latin       1.00      0.74      0.85        27
         pop       0.98      1.00      0.99        42
         r&b       1.00      0.89      0.94        27
         rap       0.00      0.00      0.00        16
        rock       0.80      1.00      0.89        56

    accuracy                           0.89       234
   macro avg       0.77      0.77      0.77       234
weighted avg       0.84      0.89      0.86       234


✅ Linear Regression Evaluation:
✅ R-squared: 0.9920727209802673
✅ Mean Squared Error: 0.029345815203809814


In [ ]:
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

drop_cols = ['track_id', 'track_name', 'artist_name', 'track_album_name', 'playlist_name', 'playlist_id']
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

df = df.dropna()

df.head()

,track_artist,lyrics,track_popularity,track_album_id,track_album_release_date,playlist_genre,playlist_subgenre,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,language,genre_encoded
0,Barbie's Cradle,Minsan pa Nang ako'y napalingon Hindi ko alam ...,41,1srJQ0njEQgd8w4XSqI4JQ,2001-01-01,rock,classic rock,0.682,0.401,2,...,1,0.0236,0.279,0.01170,0.0887,0.566,97.091,235440,tl,5
1,KARD,Get up out of my business You don't keep me fr...,65,7h5X3xhh3peIK9Y0qI5hbK,2019-09-22,pop,dance pop,0.760,0.887,9,...,1,0.0409,0.037,0.00000,0.1380,0.240,130.018,193160,en,2
2,Fuego,Fuego Uoh uoh uoh La musiica del futuroo Yeah!...,1,01nV3KuocS1NJHTsJbPkTO,2011-11-02,latin,latin pop,0.794,0.882,1,...,1,0.0361,0.114,0.00008,0.0620,0.642,117.009,188213,es,1
3,The Brothers Johnson,Get the funk out ma face Get the funk out ma f...,49,5IzEY3pod97kHrdt5Qt1RB,1976-01-01,r&b,urban contemporary,0.880,0.570,2,...,1,0.0821,0.292,0.03880,0.0674,0.791,106.035,147000,en,3
4,DREAMCATCHER,드림캐쳐 Full Moon 가사Hangul 시간을 따라 걸으면 조금씩 같이 걸으면 ...,49,6N7UA00H0IAexiQvOFGv5d,2018-01-12,edm,pop edm,0.589,0.945,7,...,1,0.1130,0.186,0.00000,0.3810,0.248,94.964,189711,et,0


In [ ]:

label_encoder = LabelEncoder()
df['genre_encoded'] = label_encoder.fit_transform(df['playlist_genre'])

X = df[['danceability', 'energy', 'loudness', 'valence', 'tempo', 'duration_ms']]  # Use only user input features
y = df['genre_encoded']

X = pd.get_dummies(X)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_lr_pred = lr.predict(X_test)

r2 = r2_score(y_test, y_lr_pred)
mse = mean_squared_error(y_test, y_lr_pred)

print(" Linear Regression Evaluation:")
print(" R-squared:", r2)
print(" Mean Squared Error:", mse)

danceability = float(input("Enter danceability (0-1): "))
energy = float(input("Enter energy (0-1): "))
loudness = float(input("Enter loudness (-60 to 0): "))
valence = float(input("Enter valence (0-1): "))
tempo = float(input("Enter tempo (BPM): "))
duration_ms = int(input("Enter duration (ms): "))

user_input = pd.DataFrame([[danceability, energy, loudness, valence, tempo, duration_ms]],
                          columns=['danceability', 'energy', 'loudness', 'valence', 'tempo', 'duration_ms'])

user_input = pd.get_dummies(user_input)

user_input_scaled = scaler.transform(user_input)

predicted_genre_lr = lr.predict(user_input_scaled)

predicted_genre_label_lr = round(predicted_genre_lr[0])

predicted_genre_label_lr = max(0, min(predicted_genre_label_lr, len(label_encoder.classes_) - 1))

predicted_genre_label = label_encoder.inverse_transform([predicted_genre_label_lr])

print("\n Predicted Genre using Linear Regression:", predicted_genre_label[0])


✅ Linear Regression Evaluation:
✅ R-squared: 0.304990261185185
✅ Mean Squared Error: 2.5728408586778246
Enter danceability (0-1): 0
Enter energy (0-1): 0.45
Enter loudness (-60 to 0): -22.5
Enter valence (0-1): 0.35
Enter tempo (BPM): 97.3
Enter duration (ms): 323452

✅ Predicted Genre using Linear Regression: rock
